In [16]:
import pexpect
import struct
import time
import sys
import codecs



IMU_MAC_ADDRESS = "40:BD:32:49:A5:C9"
UUID_DATA = "2d30c082-f39f-4ce6-923f-3484ea480596"

def unpack(payload):
    # Extract the timestamp
    tsM, tsL = struct.unpack('bb', payload[:2])
    timestamp_ms = ((tsM & 0b111111) << 7) | (tsL & 0b1111111)

    # Initialize variables
    notes = []
    i = 2  # Starting index after timestamp

    while i < len(payload):
        # Extract the command and channel
        c = struct.unpack('b', payload[i:i+1])[0]
        i += 1

        cmd = c & 0xF0
        channel = c & 0x0F

        # Assuming each note is a tuple of two bytes
        note_bytes = payload[i:i+2]
        i += 2

        # Unpack the note bytes and add to notes list
        # Modify this part based on the actual structure of your notes
        # For example, if notes are single bytes, use 'b' instead of 'bb'
        note = struct.unpack('bb', note_bytes)
        notes.append(note)

    return timestamp_ms, cmd, channel, notes


if __name__ == '__main__':
    gatt = pexpect.spawn("gatttool -b " + IMU_MAC_ADDRESS + " -I")
    gatt.sendline("connect")
    gatt.expect("Attempting to connect to 40:BD:32:49:A5:C9")
    gatt.expect("Connection successful")

    while(True):
        #gatt.sendline("char-read-uuid " + UUID_DATA)
        gatt.expect("Notification handle = 0x0009 value: ")
        gatt.expect(" \r\n")
        data = gatt.before
        timestamp_ms, cmd, channel, notes = unpack(data)
        print("Timestamp: ", timestamp_ms, " CMD: ", cmd, " CHN: ", channel, " NOTES: ", notes)

Timestamp:  4404  CMD:  32  CHN:  0  NOTES:  [(97, 102), (56, 49), (51, 100), (48, 48)]
Timestamp:  4404  CMD:  32  CHN:  0  NOTES:  [(98, 48), (56, 48), (51, 100), (48, 48)]
Timestamp:  7394  CMD:  32  CHN:  0  NOTES:  [(98, 57), (57, 49), (51, 99), (54, 48)]
Timestamp:  7394  CMD:  32  CHN:  0  NOTES:  [(98, 97), (57, 48), (51, 99), (54, 48)]
Timestamp:  7218  CMD:  32  CHN:  0  NOTES:  [(99, 51), (56, 49), (51, 99), (48, 48)]
Timestamp:  7218  CMD:  32  CHN:  0  NOTES:  [(99, 52), (56, 48), (51, 99), (48, 48)]
Timestamp:  4281  CMD:  32  CHN:  0  NOTES:  [(99, 100), (57, 49), (51, 100), (54, 48)]
Timestamp:  4281  CMD:  32  CHN:  0  NOTES:  [(99, 101), (57, 48), (51, 100), (54, 48)]
Timestamp:  7344  CMD:  32  CHN:  0  NOTES:  [(100, 55), (56, 49), (51, 100), (48, 48)]
Timestamp:  7344  CMD:  32  CHN:  0  NOTES:  [(100, 56), (56, 48), (51, 100), (48, 48)]
Timestamp:  4407  CMD:  32  CHN:  0  NOTES:  [(101, 49), (57, 49), (51, 101), (54, 48)]
Timestamp:  4407  CMD:  32  CHN:  0  NOTE

KeyboardInterrupt: 